# 05 - PLA and Backtesting

Synthetic engineering and validation evidence, not final regulatory capital.

Use this notebook with the [IMA package journey](../docs/PACKAGE_JOURNEY.md) and the executable [package demo](../examples/run_demo.py).

Raw inputs your upstream risk engine must emit: scenario P&L cubes with scenario metadata, risk-factor definitions, RFET real-price observations and evidence, stress-history series, PLA/backtesting observations, and NMRF valuation artifacts. The package dataset contract defines the committed fixture files, manifest lineage, and Arrow handoff: [`DATASET_CONTRACT.md`](../docs/DATASET_CONTRACT.md). The staged workflow is described in the [IMA package journey](../docs/PACKAGE_JOURNEY.md).

This notebook reviews the committed `capital_run_v1` PLA and backtesting vectors through the same fixture loader used by tests and examples. It does not create independent P&L data.

Regulatory anchors: Basel MAR32 PLA and backtesting concepts; U.S. NPR 2.0 proposed-rule parameters for KS-based HPL/RTPL PLA over a 250-business-day window and APL/HPL VaR backtesting at 97.5% and 99.0% cited below; EU CRR Article 325bf and 325bg as comparison context.

Prototype caution: P&L vectors are synthetic, positive = profit. VaR vectors are positive magnitudes. Outputs are not final regulatory capital or supervisory eligibility determinations.


## Public API happy path

This notebook applies PLA and backtesting gates to the committed fixture observations.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
for path in (ROOT, ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

from examples.capital_run_fixture import (
    load_capital_run_v1_fixture,
    observation_dates_from_fixture,
    policy_from_fixture,
)
from frtb_ima.backtesting import trading_desk_backtest_trace_for_policy
from frtb_ima.capital import supervisory_multiplier_for_policy
from frtb_ima.pla import pla_assessment_for_policy_with_diagnostics

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

fixture = load_capital_run_v1_fixture()
policy = policy_from_fixture(fixture)
observation_dates = observation_dates_from_fixture(fixture)
vectors = fixture.pla_bt_vectors

pla_result = pla_assessment_for_policy_with_diagnostics(
    vectors["hpl"],
    vectors["rtpl"],
    policy,
    observation_dates=observation_dates,
    run_id=str(fixture.params["run_id"]),
    desk_id=str(fixture.params["desk_id"]),
)

backtest_result = trading_desk_backtest_trace_for_policy(
    vectors["apl"],
    vectors["hpl"],
    {0.975: vectors["var_975"], 0.99: vectors["var_99"]},
    policy,
    observation_dates=observation_dates,
    run_id=str(fixture.params["run_id"]),
    desk_id=str(fixture.params["desk_id"]),
)

multiplier = supervisory_multiplier_for_policy(
    backtest_result.result.level(0.99).apl_exceptions,
    policy,
)

display(
    Markdown(
        f"Loaded `{fixture.root.name}` for `{policy.regime.value}`. "
        f"PLA window: `{pla_result.diagnostics.window_size}` observations. "
        f"Backtesting window: `{backtest_result.window_size}` observations."
    )
)


## Implementation details / math verification - PLA statistics and exception traces

The remaining cells inspect HPL/RTPL shape, Kolmogorov-Smirnov evidence, and backtesting exception traces.


## Gate Summary

The summary keeps policy-window diagnostics beside the gate result. The notebook visual sections below explain the same results using HPL/RTPL distribution shape and VaR exception geometry.

In [ ]:
def markdown_table(headers: list[str], rows: list[list[str]]) -> Markdown:
    header = "| " + " | ".join(headers) + " |"
    separator = "| " + " | ".join(["---"] * len(headers)) + " |"
    body = ["| " + " | ".join(row) + " |" for row in rows]
    return Markdown("\n".join([header, separator, *body]))


rows = [
    ["PLA KS statistic", f"{pla_result.ks_statistic:.4f}"],
    ["PLA zone", pla_result.zone],
    ["PLA window start", str(pla_result.diagnostics.start_date)],
    ["PLA window end", str(pla_result.diagnostics.end_date)],
    ["Backtesting model eligible", str(backtest_result.model_eligible)],
    [
        "Official-holiday exclusions",
        str(sum(item.official_holiday for item in backtest_result.level(0.99).observations)),
    ],
    ["Supervisory multiplier", f"{multiplier:.2f}"],
]
for level in backtest_result.result.levels:
    rows.extend(
        [
            [f"{level.confidence_level:.1%} APL exceptions", str(level.apl_exceptions)],
            [f"{level.confidence_level:.1%} HPL exceptions", str(level.hpl_exceptions)],
            [f"{level.confidence_level:.1%} level passed", str(level.level_passed)],
        ]
    )

display(markdown_table(["Measure", "Value"], rows))


## HPL and RTPL Overlay

PLA compares the distribution of HPL and RTPL over the policy window. The first chart shows time alignment; the second chart shows the daily difference that feeds the distribution divergence.

In [ ]:
pla_start = pla_result.diagnostics.start_index
pla_end = pla_result.diagnostics.end_index_exclusive
pla_dates = np.asarray(observation_dates[pla_start:pla_end], dtype=object)
hpl = np.asarray(vectors["hpl"][pla_start:pla_end], dtype=float)
rtpl = np.asarray(vectors["rtpl"][pla_start:pla_end], dtype=float)

fig, axes = plt.subplots(2, 1, figsize=(11, 6.6), sharex=True)
axes[0].plot(pla_dates, hpl, label="HPL", color="#1f77b4", linewidth=1.2)
axes[0].plot(pla_dates, rtpl, label="RTPL", color="#d62728", linewidth=1.2)
axes[0].axhline(0.0, color="#333333", linewidth=0.8, alpha=0.5)
axes[0].set_ylabel("P&L, positive = profit")
axes[0].set_title("HPL and RTPL policy window")
axes[0].legend(frameon=False, ncols=2)

axes[1].plot(pla_dates, hpl - rtpl, color="#6f4e7c", linewidth=1.1)
axes[1].axhline(0.0, color="#333333", linewidth=0.8, alpha=0.5)
axes[1].set_ylabel("HPL - RTPL")
axes[1].set_title("Daily attribution difference")
fig.tight_layout()


## KS Statistic Shape

The KS statistic is the maximum vertical distance between the empirical HPL and RTPL cumulative distribution functions. This view helps explain why the result is green, amber, or red without relying on the scalar alone.

In [ ]:
hpl_sorted = np.sort(hpl)
rtpl_sorted = np.sort(rtpl)
all_values = np.unique(np.concatenate([hpl_sorted, rtpl_sorted]))
cdf_hpl = np.searchsorted(hpl_sorted, all_values, side="right") / len(hpl_sorted)
cdf_rtpl = np.searchsorted(rtpl_sorted, all_values, side="right") / len(rtpl_sorted)
gaps = np.abs(cdf_hpl - cdf_rtpl)
gap_index = int(np.argmax(gaps))
x_star = float(all_values[gap_index])
y_low = float(min(cdf_hpl[gap_index], cdf_rtpl[gap_index]))
y_high = float(max(cdf_hpl[gap_index], cdf_rtpl[gap_index]))

fig, ax = plt.subplots(figsize=(9, 5.4))
ax.step(all_values, cdf_hpl, where="post", label="HPL empirical CDF", color="#1f77b4")
ax.step(all_values, cdf_rtpl, where="post", label="RTPL empirical CDF", color="#d62728")
ax.vlines(x_star, y_low, y_high, color="#111111", linewidth=2.0, label="Max KS gap")
ax.annotate(
    f"KS = {pla_result.ks_statistic:.3f}",
    xy=(x_star, (y_low + y_high) / 2.0),
    xytext=(x_star, min(0.95, y_high + 0.12)),
    arrowprops={"arrowstyle": "->", "color": "#111111"},
)
ax.set_xlabel("P&L value, positive = profit")
ax.set_ylabel("Empirical cumulative probability")
ax.set_title(f"PLA empirical CDF gap, zone = {pla_result.zone}")
ax.legend(frameon=False)
fig.tight_layout()


## Backtesting Exception Timeline

Backtesting flags an exception when the loss exceeds VaR, so the timeline plots loss as `-P&L` against the positive VaR threshold. A point above the VaR line would be an APL or HPL exception. The fixture currently has no exceptions, but the geometry is still visible.

In [ ]:
def trace_arrays(confidence_level: float) -> tuple[np.ndarray, ...]:
    trace = backtest_result.level(confidence_level)
    dates = np.asarray([item.observation_date for item in trace.observations], dtype=object)
    apl = np.asarray([item.apl for item in trace.observations], dtype=float)
    hpl_values = np.asarray([item.hpl for item in trace.observations], dtype=float)
    var = np.asarray([item.var_estimate for item in trace.observations], dtype=float)
    apl_flags = np.asarray([item.apl_exception for item in trace.observations], dtype=bool)
    hpl_flags = np.asarray([item.hpl_exception for item in trace.observations], dtype=bool)
    return dates, apl, hpl_values, var, apl_flags, hpl_flags


fig, axes = plt.subplots(len(backtest_result.levels), 1, figsize=(11, 7.2), sharex=True)
for ax, level_trace in zip(np.atleast_1d(axes), backtest_result.levels, strict=True):
    confidence_level = level_trace.confidence_level
    dates, apl, hpl_values, var, apl_flags, hpl_flags = trace_arrays(confidence_level)
    apl_loss = -apl
    hpl_loss = -hpl_values

    ax.plot(dates, var, color="#111111", linewidth=1.4, label=f"{confidence_level:.1%} VaR")
    ax.plot(dates, apl_loss, color="#e15759", linewidth=1.0, alpha=0.9, label="APL loss")
    ax.plot(dates, hpl_loss, color="#4e79a7", linewidth=1.0, alpha=0.9, label="HPL loss")
    ax.scatter(dates[apl_flags], apl_loss[apl_flags], color="#e15759", marker="x", s=44)
    ax.scatter(dates[hpl_flags], hpl_loss[hpl_flags], color="#4e79a7", marker="o", s=28)
    ax.axhline(0.0, color="#333333", linewidth=0.8, alpha=0.5)
    ax.set_ylabel("Loss / VaR")
    ax.set_title(
        f"{confidence_level:.1%}: APL exceptions={level_trace.result.apl_exceptions}, "
        f"HPL exceptions={level_trace.result.hpl_exceptions}"
    )
    ax.legend(frameon=False, ncols=3, loc="upper left")

fig.supxlabel("Observation date")
fig.tight_layout()


## See also

- [IMA package journey](../docs/PACKAGE_JOURNEY.md)
- [IMA dataset contract](../docs/DATASET_CONTRACT.md)
- [Client integration guide](../../../docs/CLIENT_INTEGRATION.md)
- [SBM package journey](../../frtb-sbm/docs/PACKAGE_JOURNEY.md)
- [DRC package journey](../../frtb-drc/docs/PACKAGE_JOURNEY.md)
- [RRAO package journey](../../frtb-rrao/docs/PACKAGE_JOURNEY.md)
- [CVA package journey](../../frtb-cva/docs/PACKAGE_JOURNEY.md)
